In [4]:
import pandas as pd

BASE_PATH = r'C:\Users\harsh\OneDrive\Desktop\ipl data analysis\datanew'

# 1. Load all 5 CSVs with correct dtypes
matches = pd.read_csv(
    rf'{BASE_PATH}\ipl_matches_data.csv',
    dtype={
        'team1': 'Int64',
        'team2': 'Int64',
        'toss_winner': 'Int64',
        'match_winner': 'Int64',
        'player_of_match': 'Int64',
    },
)

deliveries = pd.read_csv(
    rf'{BASE_PATH}\ball_by_ball_data.csv',
    dtype={
        'team_batting': 'Int64',
        'team_bowling': 'Int64',
    },
)

players = pd.read_csv(
    rf'{BASE_PATH}\players-data-updated.csv',
    dtype={'player_id': 'Int64'},
)

teams = pd.read_csv(
    rf'{BASE_PATH}\teams_data.csv',
    dtype={'team_id': 'Int64'},
)

aliases = pd.read_csv(
    rf'{BASE_PATH}\team_aliases.csv',
    dtype={'alias_id': 'Int64', 'team_id': 'Int64'},
)

# 2. team_map: {team_id: team_name}
team_map = dict(zip(teams['team_id'], teams['team_name']))

# 3. player_map: {player_id: player_name}
player_map = dict(zip(players['player_id'], players['player_name']))

# 4. Map team columns in matches
for col in ['team1', 'team2', 'toss_winner', 'match_winner']:
    matches[col] = matches[col].map(team_map)

# 5. Map player_of_match using player_map
matches['player_of_match'] = matches['player_of_match'].map(player_map)

# 6. Fix duplicate team name in matches and deliveries
rp_fix = {'Rising Pune Supergiant': 'Rising Pune Supergiants'}
for col in ['team1', 'team2', 'toss_winner', 'match_winner']:
    matches[col] = matches[col].replace(rp_fix)

# 7. Fix season: '2020/21' -> 2021, cast all to int
matches['season'] = (
    matches['season'].astype(str).replace({'2020/21': '2021'}).astype(int)
)

# 8. Convert match_date to datetime
matches['match_date'] = pd.to_datetime(matches['match_date'])

# 9. Map team_batting and team_bowling in deliveries + Rising Pune fix
for col in ['team_batting', 'team_bowling']:
    deliveries[col] = deliveries[col].map(team_map).replace(rp_fix)

# 10. Convert boolean-ish columns to real bools
bool_cols = ['is_wicket', 'is_wide_ball', 'is_no_ball', 'is_super_over']
bool_map = {'True': True, 'False': False, 'true': True, 'false': False,
            True: True, False: False, 1: True, 0: False, '1': True, '0': False}
for col in bool_cols:
    deliveries[col] = deliveries[col].map(bool_map).astype(bool)

# 11. Final sanity prints
print('matches.shape :', matches.shape)
print('deliveries.shape :', deliveries.shape)
print()
print(matches[['team1', 'team2', 'toss_winner', 'match_winner',
               'player_of_match', 'season']].head())
print()
print('toss_winner nulls :', matches['toss_winner'].isnull().sum())
print()
print(matches['season'].value_counts().sort_index())


matches.shape : (1212, 23)
deliveries.shape : (288226, 30)

                         team1                        team2  \
0  Royal Challengers Bangalore        Kolkata Knight Riders   
1          Sunrisers Hyderabad  Royal Challengers Bangalore   
2      Rising Pune Supergiants               Mumbai Indians   
3                Gujarat Lions        Kolkata Knight Riders   
4                 Punjab Kings      Rising Pune Supergiants   

                   toss_winner             match_winner player_of_match  \
0  Royal Challengers Bangalore    Kolkata Knight Riders     BB McCullum   
1  Royal Challengers Bangalore      Sunrisers Hyderabad    Yuvraj Singh   
2      Rising Pune Supergiants  Rising Pune Supergiants       SPD Smith   
3        Kolkata Knight Riders    Kolkata Knight Riders         CA Lynn   
4                 Punjab Kings             Punjab Kings      GJ Maxwell   

   season  
0    2008  
1    2017  
2    2017  
3    2017  
4    2017  

toss_winner nulls : 0

season
2008   

In [5]:
matches.to_csv(r'C:\Users\harsh\OneDrive\Desktop\ipl data analysis\datanew\matches_clean.csv', index=False)
deliveries.to_csv(r'C:\Users\harsh\OneDrive\Desktop\ipl data analysis\datanew\deliveries_clean.csv', index=False)
print("Saved.")

Saved.


In [6]:
matches_raw = pd.read_csv(r'C:\Users\harsh\OneDrive\Desktop\ipl data analysis\datanew\ipl_matches_data.csv')
print(matches_raw[matches_raw['team1'] == 1068][['team1','team2','match_winner']].head())
print(team_map)

Empty DataFrame
Columns: [team1, team2, match_winner]
Index: []
{np.int64(1): 'Royal Challengers Bangalore', np.int64(2): 'Sunrisers Hyderabad', np.int64(3): 'Mumbai Indians', np.int64(4): 'Rising Pune Supergiant', np.int64(5): 'Gujarat Lions', np.int64(6): 'Kolkata Knight Riders', np.int64(129): 'Chennai Super Kings', np.int64(134): 'Rajasthan Royals', np.int64(252): 'Delhi Capitals', np.int64(494): 'Punjab Kings', np.int64(614): 'Lucknow Super Giants', np.int64(615): 'Gujarat Titans', np.int64(1068): 'Deccan Chargers', np.int64(1414): 'Kochi Tuskers Kerala', np.int64(1419): 'Pune Warriors', np.int64(3604): 'Rising Pune Supergiants'}


In [7]:
print(1068 in matches_raw['team1'].values)
print(1068 in matches_raw['team2'].values)

False
False
